bronze Layer: Raw Data Ingestion

In [0]:

CATALOG = "01_bronze"
SCHEMA = "raw_data"

S3_BASE_PATH = "s3://de-global-wholesale/"

CHECKPOINT_BASE = "/Workspace/Users/narensivaprabhu04@gmail.com/Wholesale_Distributor/checkpoints/bronze/"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"  Configuration loaded")
print(f"  Catalog: {CATALOG}")
print(f"  Schema: {SCHEMA}")
print(f"  S3 Path: {S3_BASE_PATH}")
print(f"  Checkpoint: {CHECKPOINT_BASE}")

In [0]:
BRONZE_TABLES_METADATA = [
    {
        "table_name": "customers",
        "source_file": "customers.xlsx",
        "description": "Customer master data",
        "expected_rows": 305
    },
    {
        "table_name": "products",
        "source_file": "products.xlsx",
        "description": "Product catalog",
        "expected_rows": 60
    },
    {
        "table_name": "invoices",
        "source_file": "invoices.xlsx",
        "description": "Invoice headers",
        "expected_rows": 2010
    },
    {
        "table_name": "invoice_line_items",
        "source_file": "invoice_line_items.xlsx",
        "description": "Invoice line details",
        "expected_rows": 7063
    },
    {
        "table_name": "payments",
        "source_file": "payments.xlsx",
        "description": "Payment transactions",
        "expected_rows": 1605
    },
    {
        "table_name": "exchange_rates",
        "source_file": "exchange_rates.xlsx",
        "description": "Currency exchange rates",
        "expected_rows": 2193
    },
    {
        "table_name": "regions",
        "source_file": "regions.xlsx",
        "description": "Regional master data",
        "expected_rows": 4
    }
]

print(f"Metadata loaded for {len(BRONZE_TABLES_METADATA)} Bronze tables")

In [0]:
from pyspark.sql.functions import current_timestamp
import time

def ingest_bronze_table(table_metadata):
    table_name = table_metadata["table_name"]
    source_file = table_metadata["source_file"]
    description = table_metadata["description"]
    
    source_path = f"{S3_BASE_PATH}{source_file}"
    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    print(f"\n[{table_name}] Starting ingestion...")
    print(f"  Source: {source_file}")
    print(f"  Target: {target_table}")
    
    start_time = time.time()
    
    try:
        df = (spark.read
              .format("excel")
              .option("headerRows", 1)
              .load(source_path)
             )
        
      
        df = df.withColumn("ingestion_timestamp", current_timestamp())
        
        
        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        
       
        row_count = spark.table(target_table).count()
        col_count = len(spark.table(target_table).columns)
        duration = time.time() - start_time
        
        print(f" Success: {row_count:,} rows, {col_count} columns ({duration:.2f}s)")
        
        return {
            "table_name": table_name,
            "status": "SUCCESS",
            "row_count": row_count,
            "col_count": col_count,
            "duration_seconds": round(duration, 2)
        }
        
    except Exception as e:
        duration = time.time() - start_time
        print(f"  ✗ Failed: {str(e)}")
        
        return {
            "table_name": table_name,
            "status": "FAILED",
            "error": str(e),
            "duration_seconds": round(duration, 2)
        }

print("Ingestion function defined")

In [0]:
print("BRONZE LAYER INGESTION - STARTED")


ingestion_results = []

for table_metadata in BRONZE_TABLES_METADATA:
    result = ingest_bronze_table(table_metadata)
    ingestion_results.append(result)

print("BRONZE LAYER INGESTION - COMPLETED")

success_count = sum(1 for r in ingestion_results if r["status"] == "SUCCESS")
failed_count = sum(1 for r in ingestion_results if r["status"] == "FAILED")
total_rows = sum(r.get("row_count", 0) for r in ingestion_results)
total_duration = sum(r["duration_seconds"] for r in ingestion_results)

print(f"\nSummary:")
print(f"  Total Tables: {len(ingestion_results)}")
print(f"  Successful: {success_count}")
print(f"  Failed: {failed_count}")
print(f"  Total Rows: {total_rows:,}")
print(f"  Total Duration: {total_duration:.2f}s")

if failed_count > 0:
    print(f"\n  {failed_count} table(s) failed. Check logs above for details.")
else:
    print(f"\n All tables ingested successfully!")

In [0]:
import pandas as pd

summary_data = []

for result in ingestion_results:
    summary_data.append({
        "Table": result["table_name"],
        "Status": result["status"],
        "Row Count": f"{result.get('row_count', 0):,}" if result["status"] == "SUCCESS" else "N/A",
        "Columns": result.get("col_count", "N/A"),
        "Duration (s)": result["duration_seconds"]
    })

summary_df = pd.DataFrame(summary_data)


print("DETAILED INGESTION RESULTS")
print(summary_df.to_string(index=False))
print("DATA QUALITY VALIDATION")


for metadata, result in zip(BRONZE_TABLES_METADATA, ingestion_results):
    if result["status"] == "SUCCESS":
        expected = metadata.get("expected_rows", 0)
        actual = result.get("row_count", 0)
        
        if actual == expected:
            print(f" {metadata['table_name']}: Row count matches expected ({actual:,} rows)")
        else:
            variance = abs(actual - expected)
            print(f"  {metadata['table_name']}: Row count variance (Expected: {expected:,}, Actual: {actual:,}, Diff: {variance:,})")



In [0]:
import pandas as pd

tables = [
    "customers",
    "products",
    "invoices",
    "invoice_line_items",
    "payments",
    "exchange_rates",
    "regions"
]

summary_data = []

for table in tables:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        row_count = spark.table(full_table_name).count()
        col_count = len(spark.table(full_table_name).columns)
        summary_data.append({
            "Table": table,
            "Row Count": f"{row_count:,}",
            "Column Count": col_count,
            "Status": "Loaded"
        })
    except Exception as e:
        summary_data.append({
            "Table": table,
            "Row Count": "N/A",
            "Column Count": "N/A",
            "Status": f"Error: {str(e)[:50]}"
        })

summary_df = pd.DataFrame(summary_data)
print("BRONZE LAYER INGESTION SUMMARY")
print(summary_df.to_string(index=False))
print("DONE")